In [1]:
import os
import requests
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
import mlflow

RETRAIN = False
DATA_PATH = "../../02_data/mnist"
NUM_CLIENTS = 3

mlflow.set_tracking_uri("http://localhost:5100")
MLFLOW_EXPERIMENT_NAME = "edge-fl-experiment"

I0000 00:00:1787826788.122894   36317 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
#!docker pull ghcr.io/bencefarkas27/aggregator:test-auth
!docker run --name aggregator_test -d --rm -p 5100:5000 -p 9100:8000  ghcr.io/bencefarkas27/aggregator:test-auth

### Healt endpoint check

In [2]:
response = requests.get("http://localhost:9100/health")
print(f"Health check response of aggregator: {response.status_code} - {response.text}")

Health check response of aggregator: 200 - 


In [3]:
run_ids = []

def create_dummy_model(input_shape):
    model = keras.Sequential([
        keras.layers.InputLayer(shape=input_shape),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def train_client(client_id):
    # Load client data
    x_train = np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_images.npy")
    y_train = np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_labels.npy")
    # Create and train the model
    model = create_dummy_model(input_shape=x_train.shape[1:])
    model.fit(x_train, y_train, epochs=3)
    # Save the trained model weights
    model.save_weights(f"dummy_models/client_{client_id}.weights.h5")

# def mlflow_log_client_model(client_id):
#     weights_path = f"dummy_models/client_{client_id}.weights.h5"
#     x_train = np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_images.npy")
#     input_shape = x_train.shape[1:]
#     model = create_dummy_model(input_shape=input_shape)
#     model.load_weights(weights_path)
#     model.compile(optimizer='adam',
#                   loss='sparse_categorical_crossentropy',
#                   metrics=['accuracy'])
#     with mlflow.start_run(run_name=f"client_{client_id}_run") as run:
#         # Log the model to MLflow
#         mlflow.tensorflow.log_model(model, f"client_{client_id}_model")
#         run_ids.append(run.info.run_id)


def mlflow_log_client_model(client_id):
    response = requests.get(f"http://localhost:8555/client_{client_id}.weights.h5")
    response.raise_for_status()  # raises an exception on 4xx/5xx instead of silently continuing

    weights_path = f"client_{client_id}.weights.h5"
    with open(weights_path, "wb") as f:
        f.write(response.content)
    print(f"Client {client_id} weights downloaded.")
    input_shape = os.environ.get("MLFLOW_INPUT_SHAPE", (28, 28))
    model = create_dummy_model(input_shape=input_shape)
    model.load_weights(weights_path)
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    with mlflow.start_run(run_name=f"client_{client_id}_run") as run:
        # Log the model to MLflow
        mlflow.tensorflow.log_model(model, f"client_{client_id}_model")
        return run.info.run_id

def mlflow_load_client_model(as_user, experiment_name, mlflow_client_number):
    as_user("admin")
    mlflow.set_experiment(experiment_name)
    for client_id in range(mlflow_client_number):
        mlflow_log_client_model(client_id)


In [5]:
# Run these for client model training and logging to MLflow if no volume mounted client models exist for mlflow
os.environ["MLFLOW_TRACKING_USERNAME"] = "admin"
with open("../../04_deploy/aggregator/cfg/mlflow_admin.json", "r") as f:
                admin_config = json.load(f)
os.environ["MLFLOW_TRACKING_PASSWORD"] = admin_config["admin_password"]
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
for client_id in range(NUM_CLIENTS):
    if RETRAIN:
        train_client(client_id)
    mlflow_log_client_model(client_id)

Client 0 weights downloaded.


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
2026/08/27 12:34:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/27 12:34:05 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_0_run at: http://localhost:5100/#/experiments/1/runs/6bd7b157452b4858916ab1939138070b
🧪 View experiment at: http://localhost:5100/#/experiments/1


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_2', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
2026/08/27 12:34:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Client 1 weights downloaded.


...

🏃 View run client_1_run at: http://localhost:5100/#/experiments/1/runs/4aae60aeea654454ba6cc3e4ba770e3f
🧪 View experiment at: http://localhost:5100/#/experiments/1


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_4', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
2026/08/27 12:34:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Client 2 weights downloaded.


2026/08/27 12:34:20 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


🏃 View run client_2_run at: http://localhost:5100/#/experiments/1/runs/fb5b7f1b8f754f26ae69a578d638d676
🧪 View experiment at: http://localhost:5100/#/experiments/1


In [6]:
# Get run ids from experiment
experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
runs_df = mlflow.search_runs([experiment.experiment_id])
run_ids = runs_df["run_id"].tolist()
run_ids.reverse()

runs = [f"runs:/{run_ids[client_id]}/client_{client_id}_model"
        for client_id in range(NUM_CLIENTS)]
print(runs)
print(run_ids)

['runs:/6bd7b157452b4858916ab1939138070b/client_0_model', 'runs:/4aae60aeea654454ba6cc3e4ba770e3f/client_1_model', 'runs:/fb5b7f1b8f754f26ae69a578d638d676/client_2_model']
['6bd7b157452b4858916ab1939138070b', '4aae60aeea654454ba6cc3e4ba770e3f', 'fb5b7f1b8f754f26ae69a578d638d676']


### Calling do_aggregate widt fedavg

In [7]:
aggregation_config = {
    "local_model_hosts": [
        "http://localhost:5000"
        for _ in range(NUM_CLIENTS)
    ],
    "local_model_uris": [
        f"runs:/{run_ids[client_id]}/client_{client_id}_model"
        for client_id in range(NUM_CLIENTS)
    ],
    "local_data_size": [
        len(np.load(f"{DATA_PATH}/client_data/client_{client_id}_train_images.npy"))
        for client_id in range(NUM_CLIENTS)
    ],
    "aggregation_algorithm": "fedavg",
    
}

response = requests.post("http://localhost:9100/aggregate_models", json=aggregation_config)

In [8]:
print(response.json())

{'aggregated_model_host_external': 'http://localhost:5100', 'aggregated_model_host_organizational': 'http://aggregator:5000', 'aggregated_model_uri': 'models:/m-7f0ae9e3714b482bb552c8017f0bfd40', 'aggregated_model_username': 'model_user', 'aggregated_model_password': 'hiPMMENYuzajkO_vuq9owKnV3ORxRSapm7GgpNw9BAU'}


In [9]:
# Testing the client models vs the aggregated model
# Load test data
x_test = np.load(f"{DATA_PATH}/test_images.npy")
y_test = np.load(f"{DATA_PATH}/test_labels.npy")

In [10]:
# Pull the aggregated model from mlflow
try:
    # Search the experiment and retrieve the runs
    experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    models = mlflow.search_logged_models(
        experiment_ids=[experiment.experiment_id],
        order_by=[{"field_name": "creation_time", "ascending": False}],
        output_format="list"
    )
    latest_model = models[0]
    aggregated_model = mlflow.tensorflow.load_model(f"models:/{latest_model.model_id}")
    aggregated_model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    print(f"Loaded latest aggregated model: {latest_model.model_id}")

    aggregated_loss, aggregated_accuracy = aggregated_model.evaluate(x_test, y_test, verbose=0)
    print(f"Aggregated Model - Loss: {aggregated_loss}, Accuracy: {aggregated_accuracy}")
except Exception as e:
    print(f"Failed to load model: {e}")


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 10 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Loaded latest aggregated model: m-7f0ae9e3714b482bb552c8017f0bfd40
Aggregated Model - Loss: 11.153631210327148, Accuracy: 0.20409999787807465


In [11]:
client_losses = []
client_accuracies = []
for client_id in range(NUM_CLIENTS):
    # Load the client model
    client_model = create_dummy_model(input_shape=x_test.shape[1:])
    client_model.load_weights(f"dummy_models/client_{client_id}.weights.h5")
    
    # Evaluate the client model
    client_loss, client_accuracy = client_model.evaluate(x_test, y_test, verbose=0)
    print(f"Client {client_id} Model - Loss: {client_loss}, Accuracy: {client_accuracy}")
    client_losses.append(client_loss)
    client_accuracies.append(client_accuracy)

/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_7', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Client 0 Model - Loss: 82.75552368164062, Accuracy: 0.414900004863739


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_8', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Client 1 Model - Loss: 152.79278564453125, Accuracy: 0.27799999713897705


/home/barkas/miniconda3/envs/edge-test/lib/python3.12/site-packages/keras/src/saving/saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam_9', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Client 2 Model - Loss: 229.202880859375, Accuracy: 0.2930000126361847


In [12]:
print(f"Aggregator's loss is {aggregated_loss} vs clients's losses: {client_losses} and avarage client loss: {np.mean(client_losses)}")
print(f"Aggregator's accuracy is {aggregated_accuracy} vs clients's accuracies: {client_accuracies} and avarage client accuracy: {np.mean(client_accuracies)}")

Aggregator's loss is 11.153631210327148 vs clients's losses: [82.75552368164062, 152.79278564453125, 229.202880859375] and avarage client loss: 154.91706339518228
Aggregator's accuracy is 0.20409999787807465 vs clients's accuracies: [0.414900004863739, 0.27799999713897705, 0.2930000126361847] and avarage client accuracy: 0.3286333382129669


In [13]:
# Saving the aggregated model to a file
aggregated_model.save_weights(f"dummy_models/aggregated_model.weights.h5")

In [ ]:
!docker stop aggregator_test